# Стартовые условия финального мира

Ноутбук отвечает на вопрос: **какие стартовые значения мира такси делают пользу прогноза
измеримой — и при этом не подгоняют мир под ответ?** Материал — `notebooks/stop.md`
(«Мысли и план», §1–§21), помощники `notebooks/start_conditions_lab.py` и
`notebooks/action_model_lab.py`, таблицы экспериментов e2–e18 в
`data/start_conditions/scratch/*.pkl`.

Структура: диагноз старого мира (§1) → три условия пользы политики (§2) → эксперименты
e2–e18 (§3–§8) → подбор финального мира по критериям design на трёх seed (§9) →
абляции канала `price`/`heatmap`/`both` (§10) → таблица критериев с фактическими
значениями (§11) → YAML-фрагмент и ограничения (§12).

**Итог — значения финального мира (`config.final.yaml`):**

| ключ | значение | обоснование |
|---|---|---|
| `drivers.count` | 250 | колено e10 — 240; при шоках σ города 0.12 +10 машин возвращают базу в полосу 3–6 % на каждом seed (§9.1) |
| `drivers.shifts.enabled` | true | вечерний провал предложения, предсказуемый по часам |
| `drivers.dispatch.max_pickup_min` | 12 | радиус подачи: отказы от долгой подачи — 73 % необслуженных (§11) |
| `drivers.dispatch.cancel_grace_min` / `cancel_prob_per_min` / `cancel_max_prob` | 5 / 0.06 / 0.9 | параметры e10–e18 |
| `drivers.reposition.enabled` | true | отклик предложения на цену — условие пользы ценового рычага (§3) |
| `drivers.callout.enabled`, `part`, `lag_min`, `session_min` | true, 60, 60, 120 | вызов подработчиков: вред на 1–3 ч вперёд предсказуем, реактивно недоступен (e17, §19) |
| `demand.shocks.enabled`, `city_sigma` / `zone_sigma` / `day_sigma` | true, 0.12 / 0.10 / 0.08 | персистентная, не предсказуемая календарём компонента спроса (§19, §9) |
| `pricing.elasticity_alpha` | 0.5 | краткосрочная эластичность такси 0.4–0.6 (§3) |
| `ml.deficit_label` | harm | цель «вред» сдвигает кривую MLP вверх (e12, §13) |
| `ml.planning.horizons_min` | [60, 120] | длинный горизонт — там планирование обгоняет реакцию (§19) |
| `policy.channel` | both | платформа одновременно ставит цену пассажиру и сигналит водителям (§10) |

Эталонный `config.yaml` не меняется: финальный мир живёт в отдельном файле, каталогах
`data/final/` и `artifacts/final/`. Таблица критериев 1–3 с фактическими значениями —
в §11.

In [ ]:
# Настройка: корень проекта, пути и общие помощники ноутбука.
import contextlib
import io
import json
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "config.yaml").exists():  # ядро ноутбука стартует в notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from app.compare import branch_outcomes, test_window
from common.config import load_config
from ml.dataset import SqliteRowSource, world_start
from ml.planning import (
    CalendarBaseline,
    collect_planning_examples,
    latest_planning_artifact_dir,
    load_planning_artifact,
    predict_planning_windows,
    prepare_planning_data,
    save_planning_artifacts,
    score_metrics,
    train_planning,
)
from ml.train import model_version_for
from notebooks.start_conditions_lab import make_cfg, outcomes, run_world
from simulator.branches import generate_branch
from simulator.city import build_city

SCRATCH = ROOT / "data" / "start_conditions" / "scratch"
FINAL_WORK = ROOT / "data" / "start_conditions" / "final-world"
FINAL_WORK.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 80)


def table(name: str) -> pd.DataFrame:
    """Таблица эксперимента e2–e18 из scratch-каталога."""

    path = SCRATCH / f"{name}.pkl"
    if not path.exists():
        raise FileNotFoundError(f"нет таблицы {path} — восстановите data/start_conditions/scratch")
    return pd.read_pickle(path)


def add_relative(frame: pd.DataFrame, value_col: str, new_col: str) -> pd.DataFrame:
    """Добавить исход относительно базы того же seed (``lost``/``unserved``).

    «Отпугнуто» = перевезено в базе − перевезено в ветке; «спасено» = потери
    (или необслуженные) базы − потери ветки. База — строки ``pol == "a-base"``
    того же seed.
    """

    base = frame[frame.pol == "a-base"].set_index("seed")[value_col]
    out = frame.copy()
    out["base_value"] = out.seed.map(base)
    out[new_col] = out.base_value - out[value_col]
    return out

## 1. Диагноз старого мира: улучшать нечего

Эталонный `config.yaml` описывает «идеальный» мир: 380 водителей, смены, радиус подачи,
отклик и вызов выключены. В нём за тест-неделю нет потерь, подача меньше минуты, парк
загружен на пятую часть. Метка дефицита при этом срабатывает в каждой пятой
зона-пятиминутке — но не потому, что пассажиру плохо: правило
`free_drivers / max(requests, 1) < 0.5` — это снимок соотношения в конце бакета, а не
вред. Политика в таком мире может только отпугнуть пассажиров.

In [ ]:
old_cfg = load_config(ROOT / "config.yaml")
from app.compare import build_compare

old = build_compare(old_cfg, ROOT / "data" / "worlds")
old_rows = []
for branch in old["branches"]:
    o = branch["outcomes"]
    old_rows.append(
        {
            "ветка": branch["name"],
            "есть": branch["exists"],
            "поездки": o["trips"],
            "невып.": o["unfulfilled"],
            "отмены": o["cancelled"],
            "ожидание, мин": None if o["avg_wait_min"] is None else round(o["avg_wait_min"], 2),
            "выручка, млн": None if o["revenue"] is None else round(o["revenue"] / 1e6, 3),
            "дефицит, %": None
            if o["deficit_share"] is None
            else round(100 * o["deficit_share"], 1),
            "поездки к A, %": None if branch["delta"]["trips"] is None else round(branch["delta"]["trips"], 2),
        }
    )
old_table = pd.DataFrame(old_rows)
old_table

In [ ]:
# Доля метки контракта и доли «вреда» на эпохе эталонного мира (m3_epoch.db).
import sqlite3

epoch_path = ROOT / "data" / "m3_epoch.db"
conn = sqlite3.connect(f"file:{epoch_path}?mode=ro", uri=True)
total, contract, harm, label_only = conn.execute(
    """
    SELECT COUNT(*),
           SUM(CASE WHEN free_drivers * 1.0 / MAX(requests, 1) < 0.5 OR unfulfilled > 0
                    THEN 1 ELSE 0 END),
           SUM(CASE WHEN unfulfilled > 0 THEN 1 ELSE 0 END),
           SUM(CASE WHEN free_drivers * 1.0 / MAX(requests, 1) < 0.5 AND unfulfilled = 0
                    THEN 1 ELSE 0 END)
    FROM zone_metrics_5min
    """
).fetchone()
conn.close()
old_label = pd.DataFrame(
    [
        {
            "зона-пятиминуток": int(total),
            "метка контракта, %": round(100 * contract / total, 1),
            "вред (unfulfilled > 0), %": round(100 * harm / total, 1),
            "метка без вреда, %": round(100 * label_only / total, 1),
        }
    ]
)
old_label

**Вывод §1.** В старом мире (а) нет вреда, который можно убрать; (б) предложение не
откликается на цену; (в) вред и метка расходятся: 19 % метки при нуле потерь —
модели учатся предсказывать снимок соотношения, а не проблему пассажира. Поэтому
сравнение политик по потерям и выручке в нём бессмысленно: все ветки почти совпадают,
а различаются только числом отпугнутых.

## 2. Три условия, при которых политика вообще может помочь

1. **Есть вред, который можно убрать.** Необслуженные заказы (истёкшие + отказы) и
   долгое ожидание подачи. В старом мире вред = 0.
2. **Предложение откликается на цену.** Свободные водители переезжают к надбавке:
   `drivers.reposition.enabled: true`. Без отклика надбавка только срезает спрос.
3. **Надбавка не нейтральна по выручке.** Надбавка `m` срезает спрос как `m^(−α)`;
   при `α = 1` выручка на «потенциальный» заказ `λ·m^(−1)·m = λ` не меняется —
   политике нечего выигрывать. При `α = 0.5` цена частично компенсирует срезанный спрос,
   а выигрыш даёт спасённый заказ. Краткосрочная эластичность такси — 0.4–0.6, берём 0.5.

К этим условиям добавляются два механизма, которые делает политику измеримой:
**смены** (вечерний провал предложения предсказуем по часам) и **радиус подачи с
отказами** (пассажир отменяет заказ, если подача дольше терпения). Вред становится
реальным и локальным по зонам, а не общегородским исчерпанием парка.

## 3. Эксперимент 1: размер парка (e2, e4)

`e2` — первая разведка: старый мир, 380…180 водителей, `α` 1.0/0.5, отклик вкл/выкл,
политика включалась с 4-го дня. `e4` — чистый прогон базы на настоящем окне тест-недели
(дни 31–37), 9 размеров парка × 3 seed. Обе таблицы показывают одно: без потерь политика
не нужна, а с потерями — важен размер парка.

In [ ]:
e4 = table("e4")
e4_table = (
    e4.groupby("n")
    .agg(
        потери_pct=("lost_pct", "mean"),
        подача_мин=("wait", "mean"),
        подача_p90=("wait_p90", "mean"),
        загрузка_ср=("util", "mean"),
        загрузка_p95=("util_p95", "mean"),
        дефицит_pct=("deficit_pct", "mean"),
    )
    .round(2)
    .rename_axis("парк")
)
e4_table

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.2))
e4_table["потери_pct"].plot(marker="o", ax=ax)
ax.set_xlabel("парк водителей")
ax.set_ylabel("невыполненные заказы, %")
ax.set_title("e4: колено кривой потерь (среднее по seed 42/7/2026)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Колено старого мира — между 180 и 160 водителями (потери 2.0 % против 4.6 %). Но в
старом мире нет отказов от подачи и нет смен: как только включаются оба механизма, то же
колено по числу машин на линии сдвигается к **240** (e10, §4). `e2` показывает и вторую
важную вещь: без отклика водителей реактивная политика спасает заказы только ценой
срабатываний в 31 % зона-моментов, с откликом — в 17 %.

## 4. Эксперимент 2: эластичность и отклик водителей (e5)

Полный ML-конвейер (эпоха → MLP → δ-калибровка → ветки A/B/C) на кандидатах
`n ∈ {200, 180, 160, 140}`, `α ∈ {0.5, 1.0}`, отклик вкл/выкл. Среднее по seed 42 и 7.

In [ ]:
e5 = table("e5")
e5_table = (
    e5.groupby(["cfg", "pol"])
    .agg(
        перевезено=("served", "mean"),
        потери=("lost", "mean"),
        подача=("wait", "mean"),
        выручка_млн=("revenue", "mean"),
        срабатываний_pct=("fire", "mean"),
    )
    .round(2)
    .reset_index()
)
e5_table

Выводы e5: надбавка нигде не увеличила число перевезённых выше базы — парк фиксирован,
политика лишь перераспределяет машины и срезает спрос; отклик водителей резко усиливает
реактивную политику (дефицит 31 % → 17 %); при `α = 0.5` политика выигрывает по выручке,
при `α = 1.0` — почти нет. Честное сравнение требует равного бюджета срабатываний,
поэтому дальше строятся кривые «отпугнуто → спасено» (e7–e9).

## 5. Эксперимент 3: бюджет срабатываний и кривые компромисса (e6–e9)

- `e6` — равный и больший бюджет `policy.target_fire_rate` (17 % и 25 %), горизонт 30 мин,
  медленные и менее чувствительные водители. Бюджет в замкнутом контуре недобирается:
  δ под 17 % даёт в прогоне 11–12 % — модель видит, что после надбавки дефицит спадает.
- `e7` — вместо подгонки δ сетка параметров: множитель реактивной (`×1.15/1.30/1.50`),
  δ MLP/LSTM, контроль «только по потерям».
- `e8` — резкость предсказуемого фронта: `events.friday_evening.demand_mult` 1.6 и 2.0.
- `e9` — одинаковое действие (одна ступень ×M) у всех сигналов: сравнивается только
  «когда поднимать цену».

In [ ]:
e6 = table("e6")
e6_table = (
    e6[e6.pol != "a-base"]
    .groupby(["cfg", "pol"])
    .agg(срабат_pct=("fire", "mean"), потери=("lost", "mean"), подача=("wait", "mean"),
         выручка_млн=("revenue", "mean"))
    .round(2)
    .reset_index()
)
e6_table

In [ ]:
e7 = table("e7")
e7 = add_relative(e7, "served", "отпугнуто")
e7 = add_relative(e7, "lost", "спасено")
curve7 = (
    e7[e7.pol != "a-base"]
    .groupby(["pol", "param"])
    .agg(срабат_pct=("fire", "mean"), отпугнуто=("отпугнуто", "mean"),
         спасено=("спасено", "mean"), подача=("wait", "mean"), выручка_млн=("revenue", "mean"))
    .round(2)
    .reset_index()
)
curve7

In [ ]:
e8 = table("e8")
base8 = e8[e8.pol == "a-base"].set_index(["world", "seed"])[["served", "lost"]]
e8 = e8.join(base8, on=["world", "seed"], rsuffix="_base")
e8["отпугнуто"] = e8["served_base"] - e8["served"]
e8["спасено"] = e8["lost_base"] - e8["lost"]
e8_table = (
    e8[(e8.pol != "a-base") & (e8.world != "n180_fe16")]
    .groupby(["world", "pol", "param"])
    .agg(срабат_pct=("fire", "mean"), отпугнуто=("отпугнуто", "mean"),
         спасено=("спасено", "mean"))
    .round(2)
    .reset_index()
)
e8_table

Резкий фронт (e8) — единственный рычаг стартовых значений, который сдвигает сравнение в
пользу прогноза: при `demand_mult` 1.6–2.0 MLP впервые доминирует над реактивной ×1.15 —
спасает больше и почти никого не отпугивает. Но контроль «только по потерям» почти так же
хорош, поэтому e9 уравнивает действие (одна ступень ×M у всех) и показывает: разница
между сигналами — в пределах шума «отпугнуто» (±200 между seed).

In [ ]:
e9 = table("e9")
e9 = add_relative(e9, "served", "отпугнуто")
e9 = add_relative(e9, "lost", "спасено")
curve9 = (
    e9[(e9.pol != "a-base") & (e9.M.notna())]
    .groupby(["M", "pol", "param"])
    .agg(срабат_pct=("fire", "mean"), отпугнуто=("отпугнуто", "mean"),
         спасено=("спасено", "mean"), подача=("wait", "mean"), выручка_млн=("revenue", "mean"))
    .round(2)
    .reset_index()
)
curve9

Выводы e6–e9:

- Реактивная политика по метке дефицита спасает больше всех, но и отпугивает больше всех:
  её сила — в **опережающей части метки** `free/requests < 0.5`, которая загорается до
  потерь. Настоящая реактивная цена видит только уже случившиеся потери.
- MLP — точный инструмент: на мягком конце кривой (до ~150 отпугнутых) спасает
  1 100–1 400 заказов почти без отпугнутых; на жёстком конце кривая насыщается.
- LSTM на всех проверенных мирах слабее MLP.
- Единственный найденный рычаг стартовых значений, который сдвигает сравнение в пользу
  прогноза, — **резкий предсказуемый фронт** (пятничный вечер ×1.6–2.0): модель знает
  час и день недели, реакция по факту запаздывает.

## 6. Диагноз механики и новый мир: смены, радиус подачи (e10)

Механика старого мира (n=160, §10 stop.md): город — один общий пул машин, диспатч берёт
ближайшую свободную из любой зоны, потери — общегородское исчерпание парка, парк онлайн
круглосуточно. Отсюда вывод: широко срабатывающая реактивная политика выигрывает, а
точность зонального прогноза не во что конвертировать.

В коде появились `drivers.shifts` и `drivers.dispatch` (оба выключены по умолчанию, мир
побитово прежний). `e10` — сетка «парк × механизмы», 3 seed:

In [ ]:
e10 = table("e10")
e10_base = e10[e10.pol == "a-base"]
e10_table = (
    e10_base.groupby(["mech", "drivers.count"])
    .agg(не_обслужено_pct=("unserved_pct", "mean"), истекло=("lost", "mean"),
         отказы=("cancelled", "mean"), подача=("wait", "mean"),
         загрузка_ср=("util", "mean"), загрузка_p95=("util_p95", "mean"),
         дефицит_pct=("deficit_pct", "mean"))
    .round(2)
    .reset_index()
)
e10_table

- Колено — **240 водителей**: на линии в среднем ~163, как прежние 160 при круглосуточном
  парке.
- Пространственный пол ~2.7 % необслуженных остаётся и при 360 водителях: это отказы от
  подачи из соседней зоны (все дороги ≥ 5 мин, `cancel_grace_min` 5). Теперь важно, где
  стоят машины, а не только сколько их.
- Смены дают вечерний провал предложения, предсказуемый по часам; отказы составляют
  ~73 % необслуженных и уже входят в `unfulfilled` и в метку.
- Настоящая проблема — метка: на эпохе n=240 метку получают 31.7 % зона-пятиминуток,
  вред — 10.6 %, а 21.2 % помечены только по соотношению (см. §1).

## 7. Полный ML-конвейер в новом мире и цель «вред» (e11–e14)

`e11` — мир n=240, α=0.5, смены, радиус, отклик; полный конвейер (эпоха, MLP/LSTM, δ,
ветки) на 3 seed. «Спасено» считается по `unserved` (истёкшие + отказы).

In [ ]:
e11 = table("e11")
e11 = add_relative(e11, "served", "отпугнуто")
e11 = add_relative(e11, "unserved", "спасено")
e11_table = (
    e11[e11.pol != "a-base"]
    .groupby(["pol", "param"])
    .agg(срабат_pct=("fire", "mean"), отпугнуто=("отпугнуто", "mean"),
         спасено=("спасено", "mean"), не_обслужено_pct=("unserved_pct", "mean"),
         подача=("wait", "mean"), выручка_млн=("revenue", "mean"))
    .round(2)
    .reset_index()
)
e11_table

In [ ]:
e12_quality = table("e12_quality")
e12_quality.round(3)

In [ ]:
e12 = table("e12")
e12 = add_relative(e12, "served", "отпугнуто")
e12 = add_relative(e12, "unserved", "спасено")
e12_table = (
    e12[e12.pol != "a-base"]
    .groupby(["pol", "param"])
    .agg(срабат_pct=("fire", "mean"), отпугнуто=("отпугнуто", "mean"),
         спасено=("спасено", "mean"), не_обслужено_pct=("unserved_pct", "mean"),
         выручка_млн=("revenue", "mean"))
    .round(2)
    .reset_index()
)
e12_table

Цель «вред» (`unfulfilled > 0`, истёкшие + отказы) вместо правила контракта сдвигает
кривую MLP вверх на ~150–250 спасённых при равном числе отпугнутых и поднимает AP вреда
на 12–15 %. Это оправданное улучшение цели обучения, а **не** изменение мира: правило
контракта остаётся общим для реактивной ветки, F1 и сервера, меняется только таргет
обучения (`ml.deficit_label: harm`).

In [ ]:
e13 = table("e13")
e13 = add_relative(e13, "served", "отпугнуто")
e13 = add_relative(e13, "unserved", "спасено")
e13_table = (
    e13[(e13.pol != "a-base") & (e13.history == "none")]
    .groupby(["lag", "pol", "param"])
    .agg(срабат_pct=("fire", "mean"), отпугнуто=("отпугнуто", "mean"),
         спасено=("спасено", "mean"), не_обслужено_pct=("unserved_pct", "mean"),
         выручка_млн=("revenue", "mean"))
    .round(2)
    .reset_index()
)
e13_table

In [ ]:
e14 = table("e14")
e14 = add_relative(e14, "served", "отпугнуто")
e14 = add_relative(e14, "unserved", "спасено")
e14_table = (
    e14[(e14.pol != "a-base") & e14.mask_price]
    .groupby(["lag", "pol", "param"])
    .agg(срабат_pct=("fire", "mean"), отпугнуто=("отпугнуто", "mean"),
         спасено=("спасено", "mean"), не_обслужено_pct=("unserved_pct", "mean"))
    .round(2)
    .reset_index()
)
e14_table

Задержка данных `L` (§14) бьёт по реактивной политике сильнее, чем по моделям:
при `L = 15` реактивная ×1.30 теряет ~257 спасённых, MLP δ +0.05 — ~63. Обучение на
реактивной истории (§15) делает сети устойчивыми к цене, но по исходам не помогает;
маска цены убирает самоподавление, но кривую не улучшает. Значит, «самоподавление» через
признак цены работало как грубая обратная связь и было полезно.

Сводка e10–e14: реактивная политика на жёстком конце кривой сильнее во всех проверенных
вариантах; модели выигрывают на мягком конце и при задержке данных; цель «вред» —
устойчивое улучшение.

## 8. Потолок прогноза, планирование и вызов водителей (e15–e18)

**Оракул (e15).** Идеальный прогноз вреда мира без политики хуже реактивной и хуже самих
сетей на всей кривой. Причина — вред **смещается**: надбавка срезает спрос и притягивает
машины из соседей, и вред появляется там, где в мире без политики его не было (у оракула
71 % оставшегося вреда — в бакетах без надбавки, у реактивной — 24 %). Потолок не в
качестве прогноза, а в постановке контрфактической цели.

**Гибриды (e16).** Ворота «факт нехватки + прогноз», ранний взвод и удержание не обгоняют
реактивную: все сигналы лежат на одной кривой «отпугнуто → спасено» или ниже. Кривую
задаёт мир: мгновенный ценовой рычаг не выигрывает от упреждения и точности.

**Прогноз нужен там, где рычаг инерционный.** Отсюда — вызов подработчиков на линию
(`world-elastic-supply`): выход водителя происходит через `lag_min` минут, и вред на
1–3 ч вперёд реактивному вызову структурно недоступен. `e17` (§19): при задержке 20/60/120/180
мин реактивный вызов пропускает 22.5 / 44.7 / 66.3 / 83.3 % вреда, а вред на этом
горизонте предсказуем почти так же, как на 15 мин.

**Но выигрывающий прогноз — календарь.** В мире без шоков сеть лучше таблицы средних на
+0.02–0.05 AP при H=15 и ~0 при H≥120: мир почти детерминирован календарём, окну истории
нечего сообщить сверх него. Поэтому финальному миру нужна **персистентная, не
предсказуемая календарём компонента спроса** — шоки (`demand.shocks`).

In [ ]:
e17 = table("e17")
e17_table = (
    e17.pivot_table(index="H", columns="модель", values="AP").round(3).rename_axis("горизонт, мин")
)
e17_table

In [ ]:
e15 = table("e15")
e15 = add_relative(e15, "served", "отпугнуто")
e15 = add_relative(e15, "unserved", "спасено")
oracle_table = (
    e15[(e15.pol == "oracle") & (e15.min_harm == 1.0)]
    .groupby(["tag", "param"])
    .agg(срабат_pct=("fire", "mean"), отпугнуто=("отпугнуто", "mean"),
         спасено=("спасено", "mean"))
    .round(2)
    .reset_index()
)
oracle_table

In [ ]:
e16 = table("e16")
e16 = add_relative(e16, "served", "отпугнуто")
e16 = add_relative(e16, "unserved", "спасено")
hybrid_table = (
    e16[(e16.pol == "hybrid") & (e16.param == 1.30)]
    .groupby(["tag", "gate"])
    .agg(срабат_pct=("fire", "mean"), отпугнуто=("отпугнуто", "mean"),
         спасено=("спасено", "mean"))
    .round(2)
    .reset_index()
)
hybrid_table

In [ ]:
e18 = table("e18")
base11 = table("e11")
base11 = base11[base11.pol == "a-base"].set_index("seed")[["served", "unserved"]]
e18 = e18.join(base11, on="seed", rsuffix="_base")
e18["отпугнуто"] = e18["served_base"] - e18["served"]
e18["спасено"] = e18["unserved_base"] - e18["unserved"]
e18_table = (
    e18.groupby("param")
    .agg(срабат_pct=("fire", "mean"), отпугнуто=("отпугнуто", "mean"),
         спасено=("спасено", "mean"), не_обслужено_pct=("unserved_pct", "mean"),
         выручка_млн=("revenue", "mean"), средний_тариф=("mean_mult", "mean"))
    .round(2)
    .rename_axis("c")
)
e18_table

Прототип сети, учитывающей действие (e18, §20), эффект цены восстанавливает офлайн, но в
замкнутом контуре заметно хуже реактивной и хуже порога MLP «вред»: при ~350 отпугнутых
спасено 648 против 1 662 у реактивной ×1.30. Причина, вероятно, в близорукости действия и
сдвиге распределения эпохи исследования. В финальный мир механизм **не входит** — остаётся
в ноутбуке до успешной второй версии.

**Что берём в финальный мир из e2–e18:** n=240, смены, радиус подачи и отказы, отклик
водителей, α=0.5, цель «вред», вызов подработчиков с длинной задержкой, шоки спроса.
Какие именно σ шоков и какие `part`/`lag_min` — подбирается ниже по критериям приёмки.

## 9. Подбор финального мира по критериям приёмки (design, решение 3)

Стартовая точка — мир e11: n=240, смены, радиус 12, `cancel_grace_min` 5,
`cancel_prob_per_min` 0.06, `cancel_max_prob` 0.9, отклик водителей по умолчанию, α=0.5,
плюс вызов (`part` 60, `lag_min` 60, `session_min` 120) и шоки по умолчанию. Подбор меняет
только σ шоков, `part`/`lag_min` вызова и при необходимости `drivers.count`.

Критерии (порядок важности 1 > 3 > 2):

1. **База (ветка A), тест-неделя:** необслуженных 3–6 %, p95 загрузки парка 85–97 %.
2. **Сети планирования на эпохе финального мира:** AP MLP на H = 60 не ниже AP календаря
   + 0.03 (§19: без шоков выигрыш ~0).
3. **Реактивная политика в канале `both` спасает заказы** (`saved > 0`), у вызова
   ненулевые часы — рычаг работает.

Подбор идёт по трём seed (42 / 7 / 2026). Если критерий 2 недостижим при σ города ≤ 0.25,
это фиксируется как результат; тогда выбирается σ с наибольшим выигрышем при выполненном
критерии 1. Мир не подгоняется дальше, чем требует реализм.

In [ ]:
BASE_OVERRIDES = {
    "drivers.count": 250,
    "drivers.shifts.enabled": True,
    "drivers.dispatch.max_pickup_min": 12,
    "drivers.dispatch.cancel_grace_min": 5,
    "drivers.dispatch.cancel_prob_per_min": 0.06,
    "drivers.dispatch.cancel_max_prob": 0.9,
    "drivers.reposition.enabled": True,
    "drivers.callout.enabled": True,
    "drivers.callout.part": 60,
    "drivers.callout.session_min": 120,
    "pricing.elasticity_alpha": 0.5,
    "ml.deficit_label": "harm",
}
SEEDS = (42, 7, 2026)
DEFAULT_SIGMAS = {"city_sigma": 0.12, "zone_sigma": 0.10, "day_sigma": 0.08}


def world_cfg(
    seed: int,
    *,
    shocks: bool = True,
    city_sigma: float = DEFAULT_SIGMAS["city_sigma"],
    channel: str = "both",
    callout_lag: int = 60,
):
    """Конфиг финального мира с переопределениями подбора (seed мира)."""

    overrides = dict(BASE_OVERRIDES)
    overrides.update(
        {
            "demand.shocks.enabled": shocks,
            "demand.shocks.city_sigma": city_sigma,
            "policy.channel": channel,
            "drivers.callout.lag_min": callout_lag,
        }
    )
    return make_cfg(overrides, seed=seed)


def with_workdir(cfg, workdir: Path):
    """Копия конфига с путями в каталог подбора (эпоха, артефакты, планирование)."""

    ml = cfg.ml.model_copy(
        update={
            "db": str(workdir / "epoch.db"),
            "artifacts_root": str(workdir / "artifacts"),
            "planning": cfg.ml.planning.model_copy(
                update={"artifacts_root": str(workdir / "planning")}
            ),
        }
    )
    return cfg.model_copy(update={"ml": ml})


def epoch_db(seed: int, *, shocks: bool, city_sigma: float = 0.12) -> Path:
    """Эпоха 37 суток мира (кэш в ``data/start_conditions/final-world``)."""

    n = world_cfg(seed, shocks=shocks, city_sigma=city_sigma).drivers.count
    tag = f"seed={seed}__sigma={'off' if not shocks else city_sigma}__n={n}"
    workdir = FINAL_WORK / tag
    db = workdir / "epoch.db"
    if db.exists():
        return db
    workdir.mkdir(parents=True, exist_ok=True)
    cfg = with_workdir(world_cfg(seed, shocks=shocks, city_sigma=city_sigma), workdir)
    config_path = workdir / "config.yaml"
    import yaml

    config_path.write_text(
        yaml.safe_dump(cfg.model_dump(), allow_unicode=True), encoding="utf-8"
    )
    days = cfg.ml.train_days + cfg.ml.test_days
    subprocess.run(
        [
            sys.executable,
            "-m",
            "simulator",
            "--config",
            str(config_path),
            "--days",
            str(days),
            "--db",
            str(db),
        ],
        cwd=ROOT,
        check=True,
        capture_output=True,
    )
    return db


def planning_ap(seed: int, *, shocks: bool, city_sigma: float = 0.12) -> dict:
    """AP календаря и MLP планирования на H=60 для эпохи мира (кэш артефакта)."""

    tag = f"seed={seed}__sigma={'off' if not shocks else city_sigma}__n={BASE_OVERRIDES['drivers.count']}"
    workdir = FINAL_WORK / tag
    db = epoch_db(seed, shocks=shocks, city_sigma=city_sigma)
    cfg = with_workdir(world_cfg(seed, shocks=shocks, city_sigma=city_sigma), workdir)
    zones = [zone.id for zone in cfg.city.zones]
    pdir = Path(cfg.ml.planning.artifacts_root)
    if not list(pdir.glob("mlp-v1-*")):
        pdir.mkdir(parents=True, exist_ok=True)
        data = prepare_planning_data(
            str(db), cfg.ml, zones=zones, start=world_start(cfg)
        )
        result = train_planning("mlp", data, cfg.ml)
        version = model_version_for("mlp")
        save_planning_artifacts(
            pdir / version,
            model_version=version,
            family="mlp",
            model=result.model,
            data=data,
            result=result,
            ml_cfg=cfg.ml,
        )
    splits = collect_planning_examples(
        SqliteRowSource(str(db)),
        zones=zones,
        start=world_start(cfg),
        train_days=cfg.ml.train_days,
        val_days=cfg.ml.val_days,
        test_days=cfg.ml.test_days,
        window_hours=cfg.ml.window_hours,
        horizons_min=cfg.ml.planning.horizons_min,
        window_min=cfg.ml.planning.window_min,
    )
    test = splits["test"]
    calendar = CalendarBaseline.fit(splits["train"].keys, splits["train"].labels)
    cal_scores = np.stack([calendar.score(zone_id, ts) for zone_id, ts in test.keys])
    artifact = load_planning_artifact(
        latest_planning_artifact_dir(pdir, "mlp"), n_zones=max(zones) + 1
    )
    mlp_scores = predict_planning_windows(
        artifact, test.windows, [zone_id for zone_id, _ in test.keys]
    )
    k = list(cfg.ml.planning.horizons_min).index(60)
    return {
        "seed": seed,
        "AP календаря": score_metrics(test.labels[:, k], cal_scores[:, k], test.harm_sums[:, k])["ap"],
        "AP MLP": score_metrics(test.labels[:, k], mlp_scores[:, k], test.harm_sums[:, k])["ap"],
    }

### 9.1 Критерий 1: база ветки A на трёх seed

База не зависит от канала и вызова (у ветки A нет политики, а вызов запускается только
сигналом), поэтому считается дешёвым прогоном в памяти.

Сначала — σ города при стартовом парке 240: 0 (шоки выключены), 0.12 (дефолт), 0.20 и
0.25; зональная и дневная σ остаются дефолтными.

In [ ]:
def base_point(seed: int, *, n: int, shocks: bool, city_sigma: float = 0.12) -> dict:
    """Итоги тест-недели базы (ветка A) для мира с заданным парком и шоками."""

    overrides = {
        "drivers.count": n,
        "demand.shocks.enabled": shocks,
        "demand.shocks.city_sigma": city_sigma,
    }
    cfg = make_cfg({**BASE_OVERRIDES, **overrides}, seed=seed)
    result = outcomes(run_world(cfg, seed=seed), cfg)
    return {
        "парк": n,
        "seed": seed,
        "не обслужено, %": round(result["unserved_pct"], 2),
        "p95 загрузки, %": round(result["util_p95"], 1),
        "подача, мин": round(result["wait"], 2),
        "выручка, млн": round(result["revenue"], 2),
    }


crit1_sigma_rows = []
for sigma in (0.0, 0.12, 0.20, 0.25):
    for seed in SEEDS:
        row = base_point(seed, n=240, shocks=sigma > 0, city_sigma=sigma if sigma > 0 else 0.12)
        row["σ города"] = sigma
        crit1_sigma_rows.append(row)
crit1_sigma = pd.DataFrame(crit1_sigma_rows)[
    ["σ города", "парк", "seed", "не обслужено, %", "p95 загрузки, %", "подача, мин", "выручка, млн"]
]
crit1_sigma

In [ ]:
crit1_sigma_summary = (
    crit1_sigma.groupby("σ города")
    .agg(
        не_обслужено_ср=("не обслужено, %", "mean"),
        не_обслужено_min=("не обслужено, %", "min"),
        не_обслужено_max=("не обслужено, %", "max"),
        p95_min=("p95 загрузки, %", "min"),
        p95_max=("p95 загрузки, %", "max"),
    )
    .round(2)
    .rename_axis("σ города")
)
crit1_sigma_summary["критерий 1 на каждом seed"] = (
    (crit1_sigma.groupby("σ города")["не обслужено, %"].min() >= 3)
    & (crit1_sigma.groupby("σ города")["не обслужено, %"].max() <= 6)
    & (crit1_sigma.groupby("σ города")["p95 загрузки, %"].min() >= 85)
    & (crit1_sigma.groupby("σ города")["p95 загрузки, %"].max() <= 97)
)
crit1_sigma_summary

У σ = 0.20 и 0.25 необслуженность выходит за 6 % на двух seed из трёх — по приоритету
1 > 3 > 2 они отклоняются. У σ = 0.12 при парке 240 средняя необслуженность 5.26 %, но
seed 7 даёт 6.06 % — чуть выше полосы. Проверим, возвращает ли базу в полосу небольшое
увеличение парка (design разрешает менять `drivers.count`):

In [ ]:
crit1_count_rows = []
for n in (240, 250, 260):
    for seed in SEEDS:
        row = base_point(seed, n=n, shocks=True, city_sigma=0.12)
        row["σ города"] = 0.12
        crit1_count_rows.append(row)
crit1_count = pd.DataFrame(crit1_count_rows)
crit1_count

In [ ]:
crit1_count_summary = (
    crit1_count.groupby("парк")
    .agg(
        не_обслужено_ср=("не обслужено, %", "mean"),
        не_обслужено_min=("не обслужено, %", "min"),
        не_обслужено_max=("не обслужено, %", "max"),
        p95_min=("p95 загрузки, %", "min"),
        p95_max=("p95 загрузки, %", "max"),
    )
    .round(2)
    .rename_axis("парк")
)
crit1_count_summary["критерий 1 на каждом seed"] = (
    (crit1_count.groupby("парк")["не обслужено, %"].min() >= 3)
    & (crit1_count.groupby("парк")["не обслужено, %"].max() <= 6)
    & (crit1_count.groupby("парк")["p95 загрузки, %"].min() >= 85)
    & (crit1_count.groupby("парк")["p95 загрузки, %"].max() <= 97)
)
crit1_count_summary

**Выбор критерия 1:** парк **250** при σ = 0.12 — минимальное увеличение стартовых 240,
которое держит базу в полосе 3–6 % на каждом seed (4.0 / 5.7 / 4.8 %) при p95 загрузки
88–91 %. Парк 260 тоже проходит, но это лишние 10 машин без выигрыша по критериям.
Итог по критерию 1: **n = 250, σ города 0.12** (зональная 0.10 и дневная 0.08 — дефолты).

### 9.2 Критерий 2: планирование на эпохе финального мира

AP MLP на H = 60 сравнивается с AP календаря (таблица средних по зоне × типу дня ×
получасу, обученная на train-днях). Контроль — мир без шоков: по §19 выигрыш там ~0.
Шоки включают `city_sigma` 0.12 (дефолт): персистентное отклонение спроса, видимое в
недавней истории, но не в календаре. Мир — выбранный по критерию 1 (n = 250).

In [ ]:
crit2_rows = []
for shocks, city_sigma, label in (
    (False, 0.12, "без шоков"),
    (True, 0.12, "σ города 0.12"),
):
    for seed in SEEDS:
        result = planning_ap(seed, shocks=shocks, city_sigma=city_sigma)
        result["мир"] = label
        result["выигрыш"] = result["AP MLP"] - result["AP календаря"]
        crit2_rows.append(result)
crit2 = pd.DataFrame(crit2_rows)[["мир", "seed", "AP календаря", "AP MLP", "выигрыш"]].round(3)
crit2

In [ ]:
crit2_summary = (
    crit2.groupby("мир")
    .agg(
        AP_календаря=("AP календаря", "mean"),
        AP_MLP=("AP MLP", "mean"),
        выигрыш_ср=("выигрыш", "mean"),
        выигрыш_min=("выигрыш", "min"),
    )
    .round(3)
    .rename_axis("мир")
)
crit2_summary["критерий 2 (≥ +0.03 в среднем)"] = crit2_summary.выигрыш_ср >= 0.03
crit2_summary

Без шоков средний выигрыш MLP над календарём — около нуля, ровно как в §19: миру нечего
сообщить сверх календаря. С шоками σ = 0.12 средний выигрыш на выбранном парке превышает
+0.03; на seed 42 и 7 он ниже порога по отдельности (недели шумят), на seed 2026 — заметно
выше. Критерий сформулирован на среднем по трём seed и выполняется.

Увеличение σ до 0.20 поднимает выигрыш на «слабом» seed, но ломает критерий 1 на двух seed
из трёх (см. §9.1). По приоритету 1 > 3 > 2 и правилу «не подгонять мир дальше, чем
требует реализм» σ остаётся минимальным — **0.12 (дефолт шоков)**: это наименьшее
проверенное σ, при котором критерий 2 выполняется в среднем.

### 9.3 Критерий 3: вызов подработчиков и задержка `lag_min`

Реактивная ветка B в канале `both` строит и ценовой, и сигнальный контур. Проверяется
`part = 60` (резерв ~24 % штатного парка — на линии только по вызову) и две задержки
выхода: 20 мин (дефолт `drivers.callout.lag_min`) и 60 мин. Ветки прогоняются настоящим
`simulator.branches.generate_branch` (shadow=False: тени для замера исходов не нужны),
исходы считаются тем же `app.compare.branch_outcomes`, что и во вкладке «Сравнение»;
часы вызова — `callout_minutes_test / 60` из `meta.json`.

In [ ]:
def branch_tag(name: str, cfg, channel: str, lag: int) -> str:
    """Имя кэша ветки: парк, σ, seed, канал и задержка вызова входят в тег."""

    return (
        f"{name}_n{cfg.drivers.count}_s{cfg.demand.shocks.city_sigma}"
        f"_seed{cfg.seed}_{channel}_lag{lag}"
    )


def run_branch(name: str, cfg, tag: str) -> tuple[dict, dict]:
    """Прогон ветки в БД подбора и её исходы (кэш по тегу)."""

    branches_dir = FINAL_WORK / "branches"
    branches_dir.mkdir(parents=True, exist_ok=True)
    db = branches_dir / f"{tag}.db"
    if not db.exists():
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            generate_branch(
                name,
                cfg,
                seed=cfg.seed,
                days=cfg.ml.train_days + cfg.ml.test_days,
                db_path=str(db),
                shadow=False,
            )
    meta = json.loads((branches_dir / f"{tag}.meta.json").read_text(encoding="utf-8"))
    result = branch_outcomes(db, cfg, build_city(cfg).path_min, test_window(cfg))
    return result, meta


crit3_rows = []
for lag in (20, 60):
    for seed in SEEDS:
        cfg = world_cfg(seed, callout_lag=lag)
        base, _ = run_branch("a-base", cfg, branch_tag("a-base", cfg, "both", lag))
        branch, meta = run_branch(
            "b-reactive", cfg, branch_tag("b-reactive", cfg, "both", lag)
        )
        crit3_rows.append(
            {
                "lag_min": lag,
                "seed": seed,
                "база не обслужено": base["unserved"],
                "ветка не обслужено": branch["unserved"],
                "спасено": base["unserved"] - branch["unserved"],
                "отпугнуто": base["trips"] - branch["trips"],
                "часы вызова": round(meta["callout_minutes_test"] / 60, 1),
                "срабатываний цены, %": round(100 * meta["price_fire_rate"], 1),
                "срабатываний сигнала, %": round(100 * meta["signal_fire_rate"], 1),
            }
        )
crit3 = pd.DataFrame(crit3_rows)
crit3

In [ ]:
crit3_summary = (
    crit3.groupby("lag_min")
    .agg(
        спасено_ср=("спасено", "mean"),
        спасено_min=("спасено", "min"),
        отпугнуто_ср=("отпугнуто", "mean"),
        часы_вызова_ср=("часы вызова", "mean"),
        срабат_цены=("срабатываний цены, %", "mean"),
        срабат_сигнала=("срабатываний сигнала, %", "mean"),
    )
    .round(2)
    .rename_axis("lag_min")
)
crit3_summary["критерий 3 (спасено > 0 на всех seed, часы > 0)"] = (
    (crit3.groupby("lag_min")["спасено"].min() > 0)
    & (crit3.groupby("lag_min")["часы вызова"].min() > 0)
)
crit3_summary

Обе задержки дают `saved > 0` на всех seed и ненулевые часы вызова. Выбирается
`lag_min = 60` как стартовая точка design (решение 3) и более честная проверка длинного
рычага: при выходе через час сигнал сети планирования горизонта 60 работает на пределе
своего упреждения, а не с запасом. `part = 60` оставлен: резерв 25 % парка даёт заметные
часы вызова без раздувания фонда оплаты, а большее `part` в этом мире уже не увеличивает
спасённые (эффект насыщения резерва, §19).

## 10. Абляции канала: `price`, `heatmap`, `both`

Три канала `policy.channel` на выбранном мире (σ = 0.12, `part` 60, `lag_min` 60), ветка B,
три seed. `price` — политика ставит только цену пассажиру (как до change'а
`policy-driver-signal`), `heatmap` — только сигнал водителям, `both` — и цена, и сигнал.

In [ ]:
ablation_rows = []
for channel in ("price", "heatmap", "both"):
    for seed in SEEDS:
        cfg = world_cfg(seed, channel=channel)
        base, _ = run_branch("a-base", cfg, branch_tag("a-base", cfg, channel, 60))
        branch, meta = run_branch(
            "b-reactive", cfg, branch_tag("b-reactive", cfg, channel, 60)
        )
        ablation_rows.append(
            {
                "канал": channel,
                "seed": seed,
                "спасено": base["unserved"] - branch["unserved"],
                "отпугнуто": base["trips"] - branch["trips"],
                "часы вызова": round(meta["callout_minutes_test"] / 60, 1),
                "переключений цены": round(branch["price_switches"], 2),
                "выручка, млн": round(branch["revenue"], 2),
                "срабат. цены, %": round(100 * meta["price_fire_rate"], 1)
                if meta["price_fire_rate"] is not None
                else None,
                "срабат. сигнала, %": round(100 * meta["signal_fire_rate"], 1)
                if meta["signal_fire_rate"] is not None
                else None,
            }
        )
ablation = pd.DataFrame(ablation_rows)
ablation

In [ ]:
ablation_summary = (
    ablation.groupby("канал")
    .agg(
        спасено_ср=("спасено", "mean"),
        спасено_min=("спасено", "min"),
        отпугнуто_ср=("отпугнуто", "mean"),
        часы_вызова_ср=("часы вызова", "mean"),
        переключений_цены=("переключений цены", "mean"),
        выручка_млн=("выручка, млн", "mean"),
    )
    .round(2)
    .rename_axis("канал")
)
ablation_summary

Абляция показывает, что канал — не косметика, но и не «просто сумма двух рычагов»:

- У **реактивной ветки B** `price` и `both` совпадают по исходам: реактивное правило — один
  множитель, и в канале `price` он же служит сигналом водителям (шов `driver_signal_fn`
  равен тарифу). Поэтому вклад второй половины рычага виден по `heatmap`.
- `heatmap` без цены пассажиру спасает меньше заказов (таблица выше), не переключает цену
  вовсе (`price_switches = 0`) и даёт выручку ниже: цена частично компенсирует срезанный
  спрос. Зато часов вызова у него больше — без ценового отсева спрос выше, и сигнал
  срабатывает чаще.
- Различие `price` и `both` проявится на обученных ветках C/D, где цена идёт по ступеням
  M3, а сигнал — по порогу сети планирования (разные источники). Полная сетка `both` для
  четырёх веток — в отчёте `make world`.

Итог: финальный канал — **`both`** (как на реальных платформах: цена пассажиру и сигнал
водителям одновременно); `price` и `heatmap` остаются абляциями.

## 11. Таблица критериев 1–3 с фактическими значениями выбранного мира

Ниже — сводка по выбранному миру: n=250, смены, радиус 12, отказы 5/0.06/0.9, отклик,
α=0.5, вызов `part` 60 / `lag_min` 60 / `session` 120, шоки σ города 0.12 / зоны 0.10 /
суток 0.08, цель «вред», канал `both`.

In [ ]:
chosen_sigma = 0.12
chosen_n = 250
chosen_lag = 60

# Критерий 1 — база ветки A выбранного мира (§9.1, парк 250).
base_series = crit1_count[crit1_count["парк"] == chosen_n]
# Критерий 2 — средний выигрыш планирования на выбранном σ.
plan_series = crit2[crit2["мир"] == "σ города 0.12"]
# Критерий 3 — выбранная задержка вызова.
branch_series = crit3[crit3["lag_min"] == chosen_lag]

criteria = pd.DataFrame(
    [
        {
            "критерий": "1. База A: необслужено 3–6 %",
            "факт": f"{base_series['не обслужено, %'].mean():.2f} % "
            f"({base_series['не обслужено, %'].min():.2f}–{base_series['не обслужено, %'].max():.2f} %)",
            "порог": "3–6 %",
            "выполнен": bool(base_series["не обслужено, %"].between(3, 6).all()),
        },
        {
            "критерий": "1. База A: p95 загрузки 85–97 %",
            "факт": f"{base_series['p95 загрузки, %'].mean():.1f} % "
            f"({base_series['p95 загрузки, %'].min():.1f}–{base_series['p95 загрузки, %'].max():.1f} %)",
            "порог": "85–97 %",
            "выполнен": bool(
                base_series["p95 загрузки, %"].between(85, 97).all()
            ),
        },
        {
            "критерий": "2. AP MLP(H=60) − AP календаря",
            "факт": f"{plan_series['выигрыш'].mean():+.3f} "
            f"(seed 42/7/2026: {', '.join(f'{v:+.3f}' for v in plan_series['выигрыш'])})",
            "порог": "≥ +0.03 в среднем",
            "выполнен": bool(plan_series["выигрыш"].mean() >= 0.03),
        },
        {
            "критерий": "3. Реактивная в `both` спасает заказы",
            "факт": f"{branch_series['спасено'].mean():.0f} "
            f"({branch_series['спасено'].min()}–{branch_series['спасено'].max()})",
            "порог": "saved > 0 на всех seed",
            "выполнен": bool((branch_series["спасено"] > 0).all()),
        },
        {
            "критерий": "3. Часы вызванных водителей > 0",
            "факт": f"{branch_series['часы вызова'].mean():.1f} ч "
            f"({branch_series['часы вызова'].min():.1f}–{branch_series['часы вызова'].max():.1f})",
            "порог": "> 0",
            "выполнен": bool((branch_series["часы вызова"] > 0).all()),
        },
    ]
)
criteria

In [ ]:
assert criteria["выполнен"].all(), "критерии приёмки не выполнены — мир не фиксируется"
print("Все критерии 1–3 выполнены на выбранном мире.")

In [ ]:
# Сводка мира «до/после» для отчёта: база A и ветка B (both, lag 60) по трём seed.
summary_rows = []
for seed in SEEDS:
    base_row = branch_series[branch_series.seed == seed].iloc[0]
    summary_rows.append(
        {
            "seed": seed,
            "A: не обслужено": base_row["база не обслужено"],
            "B: не обслужено": base_row["ветка не обслужено"],
            "спасено": base_row["спасено"],
            "отпугнуто": base_row["отпугнуто"],
        }
    )
world_summary = pd.DataFrame(summary_rows)
world_summary

In [ ]:
from IPython.display import Markdown, display

display(
    Markdown(
        f"**Факт выбранного мира (среднее по seed 42/7/2026).** "
        f"База A теряет {base_series['не обслужено, %'].mean():.2f} % заказов "
        f"({base_series['не обслужено, %'].min():.2f}–{base_series['не обслужено, %'].max():.2f} %); "
        f"p95 загрузки {base_series['p95 загрузки, %'].mean():.1f} %. "
        f"Реактивная политика в канале `both` с вызовом `lag_min=60` спасает "
        f"{branch_series['спасено'].mean():.0f} заказов за тест-неделю "
        f"({branch_series['спасено'].min()}–{branch_series['спасено'].max()}) "
        f"и держит {branch_series['часы вызова'].mean():.0f} часов вызванных водителей. "
        f"Выигрыш планирования над календарём на H=60: "
        f"{plan_series['выигрыш'].mean():+.3f} AP в среднем."
    )
)

## 12. Финал: значения, что получают город и таксопарк, ограничения

**Рекомендуемые значения** — в таблице в начале ноутбука и в `config.final.yaml`
(комментарий файла ссылается на §9–§11 этого ноутбука). Каждое значение — из
экспериментов, а не подгонки под выигрыш прогноза:

- `drivers.count: 250` — колено e10 (240) плюс 10 машин: при шоках σ = 0.12 это
  возвращает базу в полосу 3–6 % на каждом seed (§9.1).
- `drivers.shifts.enabled: true`, `drivers.dispatch.max_pickup_min: 12`,
  `cancel_grace_min: 5`, `cancel_prob_per_min: 0.06`, `cancel_max_prob: 0.9` — смены и
  отказы; параметры e10–e18.
- `drivers.reposition.enabled: true`, `pricing.elasticity_alpha: 0.5` — условия 2–3 пользы
  ценового рычага (e2, e5).
- `drivers.callout.enabled: true`, `part: 60`, `lag_min: 60`, `session_min: 120` —
  инерционный рычаг, который предсказуем на 1–3 ч вперёд (e17, §19).
- `demand.shocks.enabled: true`, `city_sigma: 0.12`, `zone_sigma: 0.10`, `day_sigma: 0.08` —
  персистентная непредсказуемая календарём компонента; без неё выигрыш планирования
  над календарём ~0 (e17, §19).
- `ml.deficit_label: harm`, `ml.planning.horizons_min: [60, 120]`, `policy.channel: both` —
  цель обучения, горизонты и рычаг (e12, e17, e18).

**Что получают город и таксопарк за неделю** (таблица и строка «Факт» выше): база теряет
несколько процентов заказов; реактивная политика в канале `both` с вызовом подработчиков
спасает сотни заказов за тест-неделю и держит тысячи часов вызванных водителей — рычаг
реально работает. Модели планирования дают выигрыш над календарём на горизонте 1 ч.

**Ограничения:**

- Критерий 2 выполняется в среднем по трём seed; на seed 42 и 7 выигрыш ниже +0.03
  по отдельности. Это зафиксированный результат: σ сильнее ломает критерий 1, а подгонять
  мир под одну неделю нельзя.
- Кривая «отпугнуто → спасено» задаётся миром: даже идеальный прогноз (оракул) не
  обгоняет реактивную политику; ценовой рычаг мгновенный, поэтому упреждение почти ничего
  не добавляет (e15–e16).
- Сеть, учитывающая действие (e18), в мире не используется: первая версия хуже
  реактивной. Это следующий шаг исследования, а не часть финального мира.
- После смены мира нужно заново пройти конвейер (`make world CONFIG=config.final.yaml`) —
  эпоха, сети, калибровки и ветки; старые `data/` и `data/worlds` остаются
  воспроизводимыми на эталонном `config.yaml`.

**Финальный конфиг** — `config.final.yaml` в корне проекта; запуск приложения —
`make serve CONFIG=config.final.yaml`. Эталонный `config.yaml`, тесты на него и старые БД
не меняются.